# Environment check

Run this notebook top to bottom before anything else in the repository. Each check prints `PASS`,
`WARN`, or `FAIL`, and every `WARN` or `FAIL` line ends with the command or setting that fixes it.
The last cell counts the results.

The checks cover the Python version and the repository package, the libraries the reference
implementations import, the proxy key and endpoint, Langfuse, the local `BAAI/bge-small-en-v1.5`
embedding model, the cached corpora and their pinned SHA-256 digests, and whether the cache
directories are writable.

The one check that calls a model sits behind `RUN_EXPENSIVE`, which ships `False` as in every
notebook here. Set it to `True` in the next cell to send one chat completion of at most 16 tokens
to a free on-prem model through the proxy. `pytest tests/test_integration.py`, from the repository
root, makes the same call from a terminal.

A `FAIL` you cannot fix from its message is covered in
[`docs/setup.md`](../../docs/setup.md), under troubleshooting.

In [1]:
import contextlib
import importlib
import importlib.metadata
import io
import os
import platform
import sys
import tempfile
import warnings
from collections.abc import Iterator
from pathlib import Path

# Everything imported from the repository package here is standard-library
# only, so this cell runs before any extra is checked. A failure on this import
# means the environment is not synced: run the uv sync command below.
from aieng.advanced_rag.cache.paths import artifacts_dir, root
from aieng.advanced_rag.corpora import (
    cache_dir,
    cached_path,
    contract_nli,
    cuad,
    financebench,
    is_cached,
    multihop_rag,
    musique,
    serverfault,
)
from aieng.advanced_rag.corpora.download import EMBEDDING_MODEL
from aieng.advanced_rag.proxy import (
    KEY_NAMES,
    PROXY_BASE_URL,
    MissingProxyKeyError,
    find_env_file,
    is_placeholder,
    read_env_file,
    resolve_api_key,
    resolve_base_url,
)


# The one check that calls a model reads this flag. Turn it on to send one chat
# completion of at most 16 tokens through the proxy, to a model that costs nothing.
RUN_EXPENSIVE = False

#: The model the live check calls. ADVANCED_RAG_CHECK_MODEL names another.
CHECK_MODEL = os.environ.get("ADVANCED_RAG_CHECK_MODEL", "Qwen3_6-35B-A3B")

HOME = str(Path.home())
results: list[tuple[str, str]] = []


def report(status: str, name: str, detail: str = "", fix: str = "") -> None:
    """Print one check's result and keep it for the summary."""
    if status in ("PASS", "WARN", "FAIL"):
        results.append((status, name))
    line = f"{status:4}  {name}"
    if detail:
        line += f": {detail}"
    print(line)
    if fix and status != "PASS":
        print(f"      fix: {fix}")


@contextlib.contextmanager
def quiet() -> Iterator[None]:
    """Hold back the import-time warnings and progress bars some libraries print."""
    with warnings.catch_warnings(), contextlib.redirect_stderr(io.StringIO()):
        warnings.simplefilter("ignore")
        yield


def show(path: Path | str) -> str:
    """Return a path with the home directory written as ~, for committed output."""
    text = str(path)
    return "~" + text[len(HOME) :] if text.startswith(HOME) else text


SYNC = "uv sync --all-packages --all-extras --dev, from the repository root"
FETCH = "python -m aieng.advanced_rag.corpora.download --core"

## Python and the repository package

The repository pins Python 3.12 in `.python-version`. `aieng.advanced_rag` has to import from this
checkout. An editable install that points at another checkout runs that checkout's code, and every
result then describes the wrong code.

In [2]:
version = sys.version_info
if version >= (3, 12):
    report("PASS", "Python", platform.python_version())
else:
    report("FAIL", "Python", platform.python_version(), "use Python 3.12, as .python-version pins")

try:
    import aieng.advanced_rag as package

    package_dir = Path(package.__file__).resolve().parent
    repo_root = Path.cwd().resolve().parents[1]
    if repo_root in package_dir.parents:
        report("PASS", "aieng.advanced_rag", f"imports from {package_dir.relative_to(repo_root)}")
    else:
        report(
            "WARN",
            "aieng.advanced_rag",
            f"imports from {show(package_dir)}, outside this checkout",
            f"run {SYNC}, and select the advanced-rag kernel",
        )
except ImportError as error:
    repo_root = Path.cwd().resolve().parents[1]
    report("FAIL", "aieng.advanced_rag", str(error), f"run {SYNC}")

PASS  Python: 3.12.3
PASS  aieng.advanced_rag: imports from aieng-advanced-rag/aieng/advanced_rag


## Libraries the reference implementations import

Each library comes from an extra of `aieng-advanced-rag`, and `uv sync --all-packages --all-extras
--dev` installs all of them. PyTorch comes from the PyTorch CPU wheel index on Linux, so its version
ends in `+cpu` there and CUDA is not available. Nothing in the reference implementations needs a
GPU.

In [3]:
LIBRARIES = (
    ("sentence_transformers", "sentence-transformers", "local"),
    ("transformers", "transformers", "chunking"),
    ("numpy", "numpy", "retrieval"),
    ("fastembed", "fastembed", "representations"),
    ("lightrag", "lightrag-hku", "lightrag"),
    ("atlas_rag", "atlas-rag", "hipporag"),
    ("docling", "docling", "docling"),
    ("google.adk", "google-adk", "agents"),
    ("litellm", "litellm", "agents"),
    ("langfuse", "langfuse", "tracing"),
    ("openai", "openai", "lightrag"),
)

for module, distribution, extra in LIBRARIES:
    try:
        with quiet():
            importlib.import_module(module)
        report("PASS", module, importlib.metadata.version(distribution))
    except Exception as error:  # noqa: BLE001 - any import failure is the finding
        report("FAIL", module, f"{type(error).__name__}: {error}", f"the {extra} extra is missing: run {SYNC}")

try:
    torch = importlib.import_module("torch")
    build = "CPU build" if "+cpu" in torch.__version__ else "PyPI build"
    cuda = "CUDA available" if torch.cuda.is_available() else "no CUDA"
    report("PASS", "torch", f"{torch.__version__}, {build}, {cuda}")
except Exception as error:  # noqa: BLE001
    report("FAIL", "torch", f"{type(error).__name__}: {error}", f"run {SYNC}")

PASS  sentence_transformers: 6.0.0
PASS  transformers: 5.16.1
PASS  numpy: 2.3.5
PASS  fastembed: 0.8.1
PASS  lightrag: 1.5.6


PASS  atlas_rag: 0.0.5.post1
PASS  docling: 2.133.0
PASS  google.adk: 2.10.0


PASS  litellm: 1.86.2
PASS  langfuse: 4.15.6
PASS  openai: 2.54.0
PASS  torch: 2.14.1+cpu, CPU build, no CUDA


## Keys and the proxy

On a Coder workspace, the platform's `onboard` command exports `OPENAI_API_KEY` and
`OPENAI_BASE_URL` into the shell and the Jupyter server. On your own machine, the key can sit in the
repository `.env` as `VECTOR_PROXY_API_KEY`. `aieng.advanced_rag.proxy` reads the environment first
and `.env` second, and every runner in the repository reads keys through it. This cell says where
each value came from and never prints a value.

In [4]:
env_file = find_env_file()
file_values = read_env_file(env_file) if env_file else {}


def source_of(name: str) -> str | None:
    """Say where a variable is set, without reading out its value."""
    if not is_placeholder(os.environ.get(name, "")):
        return "the environment"
    if not is_placeholder(file_values.get(name, "")):
        return "the repository .env"
    return None


key_sources = [(name, source_of(name)) for name in KEY_NAMES]
found = [(name, where) for name, where in key_sources if where]
try:
    resolve_api_key()
    name, where = found[0]
    report("PASS", "proxy key", f"{name} from {where}")
except MissingProxyKeyError:
    report(
        "FAIL",
        "proxy key",
        "neither OPENAI_API_KEY nor VECTOR_PROXY_API_KEY is set",
        'on Coder, run eval "$(onboard --bootcamp-name advanced-rag-bootcamp --skip-test)" in a terminal and '
        "restart Jupyter from it; locally, put VECTOR_PROXY_API_KEY=<your_key> in the repository .env",
    )

base_url = resolve_base_url()
where = source_of("OPENAI_BASE_URL")
if base_url == PROXY_BASE_URL:
    report("PASS", "proxy endpoint", f"the Vector proxy, from {where or 'the default'}")
else:
    report(
        "WARN",
        "proxy endpoint",
        f"OPENAI_BASE_URL from {where} names an endpoint other than the Vector proxy",
        "unset OPENAI_BASE_URL unless you mean to use another OpenAI-compatible endpoint",
    )

missing = [name for name in ("LANGFUSE_PUBLIC_KEY", "LANGFUSE_SECRET_KEY") if not source_of(name)]
if missing:
    report(
        "WARN",
        "Langfuse keys",
        f"{', '.join(missing)} not set, so runs are not traced",
        "on Coder, onboard exports them; locally, add them to the repository .env",
    )
else:
    report("PASS", "Langfuse keys", "both set")

PASS  proxy key: VECTOR_PROXY_API_KEY from the environment
PASS  proxy endpoint: the Vector proxy, from the default
PASS  Langfuse keys: both set


## One chat completion through the proxy

The call asks for two letters with thinking off. The default model is a reasoning model, and with
thinking on it can spend the whole token budget before it answers, which would read as a broken
proxy.

In [5]:
if not RUN_EXPENSIVE:
    report("SKIP", "proxy completion", "RUN_EXPENSIVE is False; set it to True and re-run to call the proxy")
else:
    try:
        from aieng.advanced_rag.reasoning import THINKING_OFF
        from openai import OpenAI

        client = OpenAI(base_url=resolve_base_url(), api_key=resolve_api_key(), max_retries=1, timeout=60.0)
        response = client.chat.completions.create(
            model=CHECK_MODEL,
            messages=[{"role": "user", "content": "Reply with exactly: OK"}],
            max_tokens=16,
            temperature=0,
            extra_body=THINKING_OFF,
        )
        text = (response.choices[0].message.content or "").strip()
        if text:
            report("PASS", "proxy completion", f"{CHECK_MODEL} answered {text[:20]!r}")
        else:
            report("FAIL", "proxy completion", f"{CHECK_MODEL} returned no text", "see docs/setup.md, troubleshooting")
    except Exception as error:  # noqa: BLE001 - the error type is the finding
        report(
            "FAIL",
            "proxy completion",
            type(error).__name__,
            "a 401 means the key is wrong; a 403 means the key cannot use this model, so set "
            "ADVANCED_RAG_CHECK_MODEL to one it can; see docs/setup.md, troubleshooting",
        )

SKIP  proxy completion: RUN_EXPENSIVE is False; set it to True and re-run to call the proxy


## Langfuse

Langfuse records what a run called and what each call cost. The check asks Langfuse to accept the
keys and makes no model call.

In [6]:
if missing:
    report("SKIP", "Langfuse auth", "keys not set")
else:
    for name, value in file_values.items():
        if name.startswith("LANGFUSE_"):
            os.environ.setdefault(name, value)
    try:
        from langfuse import get_client

        if get_client().auth_check():
            report("PASS", "Langfuse auth", "the keys are accepted")
        else:
            report(
                "FAIL", "Langfuse auth", "the keys are rejected", "check LANGFUSE_PUBLIC_KEY and LANGFUSE_SECRET_KEY"
            )
    except Exception as error:  # noqa: BLE001
        report(
            "FAIL",
            "Langfuse auth",
            type(error).__name__,
            "check both keys, and set LANGFUSE_HOST or LANGFUSE_BASE_URL outside the default region",
        )

PASS  Langfuse auth: the keys are accepted


## The local embedding model

Every dense arm embeds locally with `BAAI/bge-small-en-v1.5` on the CPU. The model loads from the
Hugging Face cache only, so this cell never downloads anything, and a 384-dimension vector for one
sentence shows it works.

In [7]:
try:
    with quiet():
        from sentence_transformers import SentenceTransformer

    with quiet():
        model = SentenceTransformer(EMBEDDING_MODEL, device="cpu", local_files_only=True)
        vector = model.encode(["Either party may terminate on ninety days written notice."])[0]
    if len(vector) == 384:
        report("PASS", EMBEDDING_MODEL, "loads from the local cache, 384 dimensions")
    else:
        report("FAIL", EMBEDDING_MODEL, f"returned {len(vector)} dimensions, expected 384", FETCH)
except Exception as error:  # noqa: BLE001
    report("FAIL", EMBEDDING_MODEL, f"not in the local cache ({type(error).__name__})", FETCH)

PASS  BAAI/bge-small-en-v1.5: loads from the local cache, 384 dimensions


## The corpora

The core files are the ones every reference implementation starts from, and `scripts/setup.sh`
fetches them on each workspace start. Each one is checked against its pinned SHA-256 without
downloading anything. The larger files that only one RI reads are listed as `INFO` and do not count
toward the summary.

In [8]:
CORE_SOURCES = (
    cuad.CUAD_TEST,
    contract_nli.CONTRACT_NLI_TEST,
    multihop_rag.MULTIHOP_QUERIES,
    musique.MUSIQUE_QUESTIONS,
    musique.MUSIQUE_CORPUS,
    financebench.FINANCEBENCH_OPEN_SOURCE,
    financebench.FINANCEBENCH_DOCUMENT_INFORMATION,
)

print(f"corpus cache: {show(cache_dir())}")
for source in CORE_SOURCES:
    if is_cached(source):
        report("PASS", source.name, "cached and matches its pin")
    elif cached_path(source).exists():
        report(
            "FAIL",
            source.name,
            "cached but does not match its pin",
            f"delete {show(cached_path(source))}, then {FETCH}",
        )
    else:
        report("WARN", source.name, "not fetched yet", FETCH)

pdfs = cache_dir() / "financebench-pdfs"
pdf_count = len(list(pdfs.glob("*.pdf"))) if pdfs.is_dir() else 0
report(
    "INFO",
    "FinanceBench filings (RI #3)",
    f"{pdf_count} of 84 cached; fetch with: python -m aieng.advanced_rag.corpora.download 3",
)
slice_file = cache_dir() / f"serverfault-slice-{serverfault.SLICE_10K_QUESTIONS}.json"
state = "cached" if slice_file.is_file() else "not fetched"
report("INFO", "ServerFault slice (RI #4)", f"{state}; fetch with: python -m aieng.advanced_rag.corpora.download 4")

corpus cache: ~/.cache/aieng-advanced-rag/corpora
PASS  cuad-test: cached and matches its pin
PASS  contract-nli-test: cached and matches its pin
PASS  multihop-rag-queries: cached and matches its pin
PASS  musique-questions: cached and matches its pin
PASS  musique-corpus: cached and matches its pin
PASS  financebench-open-source: cached and matches its pin
PASS  financebench-document-information: cached and matches its pin
INFO  FinanceBench filings (RI #3): 0 of 84 cached; fetch with: python -m aieng.advanced_rag.corpora.download 3
INFO  ServerFault slice (RI #4): not fetched; fetch with: python -m aieng.advanced_rag.corpora.download 4


## Writable directories

Every cache in the package hangs off one root, `~/.cache/aieng-advanced-rag` unless
`ADVANCED_RAG_CACHE_DIR` moves it. The Hugging Face cache holds the embedding model. A run that
cannot write to either fails partway through, so each one is tested with a temporary file.

In [9]:
hf_home = Path(
    os.environ.get("HF_HOME") or Path(os.environ.get("XDG_CACHE_HOME") or Path.home() / ".cache") / "huggingface"
)
DIRECTORIES = (
    ("cache root", root()),
    ("corpus cache", cache_dir()),
    ("artifact cache", artifacts_dir()),
    ("Hugging Face cache", hf_home),
)

for label, directory in DIRECTORIES:
    try:
        directory.mkdir(parents=True, exist_ok=True)
        with tempfile.NamedTemporaryFile(dir=directory):
            pass
        report("PASS", label, f"{show(directory)} is writable")
    except OSError as error:
        report(
            "FAIL",
            label,
            f"{show(directory)}: {error.strerror}",
            "free disk space, or point ADVANCED_RAG_CACHE_DIR or HF_HOME at a writable directory",
        )

PASS  cache root: ~/.cache/aieng-advanced-rag is writable
PASS  corpus cache: ~/.cache/aieng-advanced-rag/corpora is writable
PASS  artifact cache: ~/.cache/aieng-advanced-rag/artifacts is writable
PASS  Hugging Face cache: ~/.cache/huggingface is writable


## Summary

In [10]:
counts = {status: sum(1 for s, _ in results if s == status) for status in ("PASS", "WARN", "FAIL")}
print(f"{counts['PASS']} passed, {counts['WARN']} warnings, {counts['FAIL']} failed")
failed = [name for status, name in results if status == "FAIL"]
warned = [name for status, name in results if status == "WARN"]
if failed:
    print("Fix before going on: " + ", ".join(failed))
elif warned:
    print("Ready. The warnings name optional setup: " + ", ".join(warned))
else:
    print("Ready. Continue with implementations/regulatory_policy_qa/01_harness_and_baseline.ipynb.")
if not RUN_EXPENSIVE:
    print("The proxy completion did not run. Set RUN_EXPENSIVE = True to include it.")

30 passed, 0 warnings, 0 failed
Ready. Continue with implementations/regulatory_policy_qa/01_harness_and_baseline.ipynb.
The proxy completion did not run. Set RUN_EXPENSIVE = True to include it.
